In [1]:
from pathlib import Path

import io
import onnx
import onnxruntime.training.onnxblock as onnxblock
import torch
import timm

No CUDA runtime is found, using CUDA_HOME='/usr/local/cuda-12.3'
/home/mateusz/miniconda3/envs/odt/lib/python3.11/site-packages/onnxruntime/capi/onnxruntime_validation.py:114: UserWarning: WARNING: failed to get cudart_version from onnxruntime build info.
  warnings.warn("WARNING: failed to get cudart_version from onnxruntime build info.")
/home/mateusz/miniconda3/envs/odt/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Create a Classifier instance.
device = "cpu"
batch_size = 64
num_classes = 10
channels = 3
img_h, img_w = 32, 32

pt_model = timm.create_model('resnet18', pretrained=False, num_classes=num_classes, in_chans=channels, norm_layer='frozenbatchnorm2d')
pt_model

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): FrozenBatchNorm2d(64, eps=1e-05)
  (act1): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): FrozenBatchNorm2d(64, eps=1e-05)
      (drop_block): Identity()
      (act1): ReLU(inplace=True)
      (aa): Identity()
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): FrozenBatchNorm2d(64, eps=1e-05)
      (act2): ReLU(inplace=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): FrozenBatchNorm2d(64, eps=1e-05)
      (drop_block): Identity()
      (act1): ReLU(inplace=True)
      (aa): Identity()
      (conv2): Conv2d(64, 64, kernel_size=(3, 3),

### Generating forward-only graph

In [3]:
# Generate a random input.
model_inputs = (torch.randn(batch_size, channels, img_h, img_w, device=device),)

model_outputs = pt_model(*model_inputs)
if isinstance(model_outputs, torch.Tensor):
    model_outputs = [model_outputs]
    
input_names = ["input"]
output_names = ["output"]
dynamic_axes = {"input": {0: "batch_size"}, "output": {0: "batch_size"}}

f = io.BytesIO()
torch.onnx.export(
    pt_model,
    model_inputs,
    f,
    input_names=input_names,
    output_names=output_names,
    opset_version=17,
    do_constant_folding=False,
    training=torch.onnx.TrainingMode.TRAINING,
    dynamic_axes=dynamic_axes,
    export_params=True,
    keep_initializers_as_inputs=False,
)

onnx_model = onnx.load_model_from_string(f.getvalue())

### Generating training graph

Generating artifacts based on forward-only graph and the specified loss function using onnxblock library

In [4]:
# Creating a class with a Loss function.
class CustomTrainingBlock(onnxblock.TrainingBlock):
    def __init__(self):
        super(CustomTrainingBlock, self).__init__()
        self.loss = onnxblock.loss.CrossEntropyLoss()

    def build(self, output_name):
        return self.loss(output_name), output_name

In [5]:
# Build the onnx model with loss
training_block = CustomTrainingBlock()
for param in onnx_model.graph.initializer:
    print(param.name)
    # if 'conv' in param.name or 'fc' in param.name:
    training_block.requires_grad(param.name, True)

# Building training graph and eval graph.
model_params = None
with onnxblock.base(onnx_model):
    _ = training_block(*[output.name for output in onnx_model.graph.output])
    training_model, eval_model = training_block.to_model_proto()
    model_params = training_block.parameters()

# Building the optimizer graph
optimizer_block = onnxblock.optim.AdamW()
with onnxblock.empty_base() as accessor:
    _ = optimizer_block(model_params)
    optimizer_model = optimizer_block.to_model_proto()

2024-04-08 11:35:22,045 root [DEBUG] - Building training block CustomTrainingBlock
2024-04-08 11:35:22,046 root [DEBUG] - Building block: CrossEntropyLoss


conv1.weight
bn1.weight
bn1.bias
bn1.running_mean
bn1.running_var
layer1.0.conv1.weight
layer1.0.bn1.weight
layer1.0.bn1.bias
layer1.0.bn1.running_mean
layer1.0.bn1.running_var
layer1.0.conv2.weight
layer1.0.bn2.weight
layer1.0.bn2.bias
layer1.0.bn2.running_mean
layer1.0.bn2.running_var
layer1.1.conv1.weight
layer1.1.bn1.weight
layer1.1.bn1.bias
layer1.1.bn1.running_mean
layer1.1.bn1.running_var
layer1.1.conv2.weight
layer1.1.bn2.weight
layer1.1.bn2.bias
layer1.1.bn2.running_mean
layer1.1.bn2.running_var
layer2.0.conv1.weight
layer2.0.bn1.weight
layer2.0.bn1.bias
layer2.0.bn1.running_mean
layer2.0.bn1.running_var
layer2.0.conv2.weight
layer2.0.bn2.weight
layer2.0.bn2.bias
layer2.0.bn2.running_mean
layer2.0.bn2.running_var
layer2.0.downsample.0.weight
layer2.0.downsample.1.weight
layer2.0.downsample.1.bias
layer2.0.downsample.1.running_mean
layer2.0.downsample.1.running_var
layer2.1.conv1.weight
layer2.1.bn1.weight
layer2.1.bn1.bias
layer2.1.bn1.running_mean
layer2.1.bn1.running_var
lay

2024-04-08 11:35:22,627 root [DEBUG] - Building gradient graph for training block CustomTrainingBlock
2024-04-08 11:35:22.639867261 [I:onnxruntime:Default, constant_sharing.cc:256 ApplyImpl] Total shared scalar initializer count: 117
2024-04-08 11:35:22,671 root [DEBUG] - The loss output is onnx::loss::2. The gradient graph will be built starting from onnx::loss::2_grad.
2024-04-08 11:35:22.652083938 [I:onnxruntime:Default, graph.cc:3556 CleanUnusedInitializersAndNodeArgs] Removing initializer 'ortshared_1_0_1_1'. It is no longer used by any node.
2024-04-08 11:35:22,719 root [DEBUG] - Adding gradient accumulation nodes for training block CustomTrainingBlock
2024-04-08 11:35:22,728 root [DEBUG] - Building forward block AdamW
2024-04-08 11:35:22,730 root [DEBUG] - Building block: AdamWOptimizer


In [6]:
# artifacts path
artifacts_path = Path('./artifacts')
artifacts_path.mkdir(parents=True, exist_ok=True)

# Saving all the files to use them later for the training.
onnxblock.save_checkpoint(training_block.parameters(), artifacts_path / 'checkpoint')

ir_version = 9

training_model.ir_version = ir_version
onnx.save(training_model, artifacts_path / 'training_model.onnx')

optimizer_model.ir_version = ir_version
onnx.save(optimizer_model, artifacts_path / 'optimizer_model.onnx')

eval_model.ir_version = ir_version
onnx.save(eval_model, artifacts_path / 'eval_model.onnx')